In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from analytics.forecasting.base import SimpleForecaster
from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_BASELINE,
    ARTIFACTS_DIR,
    TEST_TICKERS,
)
SPAN = 20

In [2]:
# Load pool: all tickers stacked into one DataFrame
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL    1256
JNJ     1256
JPM     1256
MSFT    1256
SPY     1256
WMT     1256
dtype: int64


,timestamp,symbol,close
0,2021-03-04,AAPL,120.129997
1,2021-03-05,AAPL,121.419998
2,2021-03-08,AAPL,116.360001
3,2021-03-09,AAPL,121.089996
4,2021-03-10,AAPL,119.980003
5,2021-03-11,AAPL,121.959999
6,2021-03-12,AAPL,121.029999
7,2021-03-15,AAPL,123.989998
8,2021-03-16,AAPL,125.570000
9,2021-03-17,AAPL,124.760002


In [3]:
# 21-day-ahead direct forecast: 60-step test window, rolling by 1 week; average error across mini-windows
def get_forecast_baseline(context: pd.Series, horizon: int):
    model = SimpleForecaster(span=SPAN, confidence_level=0.95)
    model.fit(context)
    fc = model.forecast(periods=horizon)
    point = (fc.get("point_forecast") or []) if fc else []
    return [float(x) for x in point] if point else []

model_name = "baseline"
all_preds = []
for sym in TEST_TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    close_ser = grp.set_index("timestamp")["close"]
    if isinstance(close_ser, pd.DataFrame):
        close_ser = close_ser.iloc[:, 0] if close_ser.shape[1] == 1 else close_ser[sym] if sym in close_ser.columns else close_ser.iloc[:, 0]
    prices = close_ser.astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_TRAIN_BASELINE:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_TRAIN_BASELINE,
        get_forecast_fn=get_forecast_baseline,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_baseline = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_baseline.groupby("symbol").size() if not pred_baseline.empty else "No predictions.")
pred_baseline.head()

symbol
MSFT    126
SPY     126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-05,483.160004,491.7402,0,MSFT
1,2025-12-08,491.019989,491.7402,0,MSFT
2,2025-12-09,492.019989,491.7402,0,MSFT
3,2025-12-10,478.559998,491.7402,0,MSFT
4,2025-12-11,483.470001,491.7402,0,MSFT


In [4]:
# Metrics per symbol and overall: averaged over rolling mini-windows (MAE, RMSE, MAPE_%)
metrics_rows = []
for sym in pred_baseline["symbol"].unique():
    sub = pred_baseline[pred_baseline["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_baseline)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_baseline_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_baseline_pool.parquet")

      model   symbol        MAE       RMSE    MAPE_%
0  baseline     MSFT  28.319475  33.040432  6.714951
1  baseline      SPY   7.066915   7.851065  1.025719
2  baseline  overall  17.693195  24.615319  3.870335
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_baseline_pool.parquet
